# Semiconductor Yield Intelligence  
## 03 – CNN Model Training Pipeline  

---

### 🎯 Objective

Train a lightweight Convolutional Neural Network (CNN) to classify wafer defect patterns.

This notebook:

- Loads processed training data
- Defines clean model architecture
- Trains model with controlled compute usage
- Visualizes training performance
- Saves model artifact

---

## 🏗 Why CNN?

Wafer maps are spatial 2D grids.  
CNNs are designed to learn:

- Local spatial correlations
- Geometric defect structures
- Clustered failure patterns

This makes CNN the correct modeling choice.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import boto3
import os
from torch.utils.data import Dataset, DataLoader

## 🔹 Step 1 – Load Training Data from S3

In [2]:
S3_BUCKET = "sagemaker-us-east-1-702452513784"
S3_PREFIX = "wafer-project/processed"

LOCAL_DATA_DIR = "/home/sagemaker-user/MLops-Project-AAI-540/data"
s3 = boto3.client("s3")

files = ["X_train.npy", "y_train.npy"]

for file in files:
    s3.download_file(
        S3_BUCKET,
        f"{S3_PREFIX}/{file}",
        f"{LOCAL_DATA_DIR}/{file}"
    )

X_train = np.load(f"{LOCAL_DATA_DIR}/X_train.npy")
y_train = np.load(f"{LOCAL_DATA_DIR}/y_train.npy")

# 🟢 Dataset class
## 🔹 Step 2 – Create PyTorch Dataset Class

We wrap NumPy arrays into a PyTorch Dataset object to:

- Enable batching
- Enable shuffling
- Prepare data for GPU/CPU training

In [4]:
class WaferDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx].unsqueeze(0), self.y[idx]

# 🟢 DataLoader
## 🔹 Step 3 – DataLoader Configuration

Batch size is kept small to:

- Reduce memory usage
- Lower AWS compute cost
- Maintain stable gradients

In [5]:
train_loader = DataLoader(
    WaferDataset(X_train, y_train),
    batch_size=32,
    shuffle=True
)

# 🟢 Model Definition
## 🔹 Step 4 – Define CNN Architecture

Model is intentionally lightweight:

- 1 convolution layer
- 1 fully connected hidden layer
- Drop complexity to reduce compute cost

In [6]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.conv1 = nn.Conv2d(1,16,3,padding=1)
        self.pool = nn.MaxPool2d(2,2)
        self.fc1 = nn.Linear(16*16*16,64)
        self.fc2 = nn.Linear(64,num_classes)

    def forward(self,x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = x.view(x.size(0),-1)
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

# 🟢 Training Loop
## 🔹 Step 5 – Training Loop

Training is intentionally short (5 epochs) to:

- Minimize AWS cost
- Demonstrate full pipeline
- Avoid overfitting small dataset

In [7]:
model = SimpleCNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 5  # Low-cost training

for epoch in range(EPOCHS):
    total_loss = 0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

Epoch 1, Loss: 1.6343
Epoch 2, Loss: 1.6105
Epoch 3, Loss: 1.6057
Epoch 4, Loss: 1.6092
Epoch 5, Loss: 1.6024


# 🟢 Save Model

In [8]:
torch.save(model.state_dict(), "../data/model.pt")
print("Model saved successfully.")

Model saved successfully.


## 🔹 Training Loss Visualization

Visualizing convergence ensures:

- Stable learning
- No exploding gradients
- Controlled optimization